In [3]:
# ============================================================
# CodeAlpha — Task 4: Sentiment Analysis
# Approach: NLP + lexicon-based sentiment classification (VADER)
# Dataset: Tweets (auto-downloaded) with built-in fallback
# Word clouds are OPTIONAL — script runs even if wordcloud is missing
# ============================================================

# ---------- 1. Imports ----------
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import warnings
from collections import Counter

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['figure.max_open_warning'] = 0

IMG_DIR = "images"
DATA_DIR = "data"
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ---------- 2. NLTK setup ----------
import nltk
for pkg in ['vader_lexicon', 'stopwords', 'punkt', 'punkt_tab']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords

# ---------- 3. OPTIONAL: WordCloud ----------
try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
    print("✅ wordcloud is installed — word clouds will be generated.")
except ImportError:
    WORDCLOUD_AVAILABLE = False
    print("⚠ wordcloud NOT installed — word clouds will be skipped.")
    print("  To enable: pip install wordcloud")

sia = SentimentIntensityAnalyzer()
STOPWORDS = set(stopwords.words('english'))

# ============================================================
# 4. LOAD DATASET
# ============================================================
print("\n" + "=" * 60)
print("LOADING DATASET")
print("=" * 60)

df = None
try:
    url = "https://raw.githubusercontent.com/ywchiu/riii/master/data/hillary_tweets.csv"
    df = pd.read_csv(url)
    if 'text' not in df.columns:
        raise ValueError("'text' column not found")
    df = df[['text']].dropna()
    print(f"✅ Loaded from URL: {len(df)} tweets")
except Exception as e:
    print(f"⚠ URL load failed ({e}). Using built-in sample.")
    sample_texts = [
        "I absolutely love this product! Best purchase ever.",
        "This is the worst experience I've ever had. Terrible service.",
        "The movie was okay, nothing special but not bad either.",
        "Amazing quality and fast delivery. Highly recommend!",
        "Disappointed with the customer support. Very rude.",
        "Just got my order, everything looks great so far.",
        "Not worth the price. Would not buy again.",
        "Fantastic! Exceeded all my expectations.",
        "It's fine. Does the job, nothing more.",
        "Horrible packaging, product arrived damaged.",
        "Super happy with this purchase, will buy again!",
        "Meh. Expected more based on the reviews.",
        "Outstanding quality and fantastic customer service!",
        "Waste of money. Completely useless.",
        "Pretty good overall, minor issues but works well.",
        "I'm extremely satisfied with this. Great value!",
        "The worst customer experience I've ever had.",
        "Decent product, but shipping took way too long.",
        "Absolutely love it! Perfect gift for my mom.",
        "Broken on arrival. Requested a refund immediately.",
        "This exceeded my expectations. Amazing!",
        "Would not recommend. Poor quality materials.",
        "Good product for the price. Happy with it.",
        "Terrible. Nothing works as advertised.",
        "Best thing I've bought all year. 10/10!",
        "Not bad, but could be better.",
        "Excellent service and quick response time!",
        "Very disappointing. Not what I expected.",
        "Love love love this! So happy with my purchase.",
        "Average product. Nothing to write home about.",
    ]
    df = pd.DataFrame({'text': sample_texts})
    print(f"✅ Using built-in sample: {len(df)} texts")

# ============================================================
# 5. CLEAN TEXT
# ============================================================
print("\n" + "=" * 60)
print("TEXT CLEANING")
print("=" * 60)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)      # URLs
    text = re.sub(r'@\w+', '', text)                          # mentions
    text = re.sub(r'#', '', text)                             # hashtag symbol
    text = re.sub(r'[^A-Za-z\s]', ' ', text)                  # keep letters
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
print(f"Rows after cleaning: {len(df)}")
print(df[['text', 'clean_text']].head())

# ============================================================
# 6. SENTIMENT SCORING (VADER)
# ============================================================
print("\n" + "=" * 60)
print("SENTIMENT SCORING")
print("=" * 60)

def get_sentiment(text):
    scores = sia.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        label = 'Positive'
    elif compound <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'
    return pd.Series([scores['pos'], scores['neu'], scores['neg'],
                      compound, label])

df[['pos', 'neu', 'neg', 'compound', 'sentiment']] = df['clean_text'].apply(get_sentiment)

print(df['sentiment'].value_counts())
print("\nSentiment distribution (%):")
print((df['sentiment'].value_counts(normalize=True) * 100).round(2))

df.to_csv(f"{DATA_DIR}/sentiment_results.csv", index=False)

# ============================================================
# 7. VISUALIZATIONS
# ============================================================
print("\n" + "=" * 60)
print("GENERATING VISUALIZATIONS")
print("=" * 60)

SENT_COLORS = {'Positive': '#2ecc71', 'Neutral': '#95a5a6', 'Negative': '#e74c3c'}
counts = df['sentiment'].value_counts().reindex(
    ['Positive', 'Neutral', 'Negative']).fillna(0)

# ---------- CHART 1 — Sentiment Distribution (Donut) ----------
fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
       colors=[SENT_COLORS[k] for k in counts.index],
       startangle=90,
       wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2),
       textprops={'fontsize': 12, 'weight': 'bold'})
ax.set_title("Overall Sentiment Distribution", fontsize=15, weight='bold', pad=20)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/01_sentiment_donut.png", bbox_inches='tight')
plt.close()

# ---------- CHART 2 — Sentiment Count Bar ----------
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(counts.index, counts.values,
              color=[SENT_COLORS[k] for k in counts.index], edgecolor='black')
ax.set_title("Number of Texts by Sentiment", fontsize=15, weight='bold')
ax.set_ylabel("Count", fontsize=12)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(counts)*0.01,
            int(v), ha='center', fontsize=12, weight='bold')
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/02_sentiment_count.png", bbox_inches='tight')
plt.close()

# ---------- CHART 3 — Compound Score Distribution ----------
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x='compound', hue='sentiment', bins=30,
             palette=SENT_COLORS, alpha=0.7, ax=ax)
ax.axvline(0.05, color='gray', linestyle='--', alpha=0.6)
ax.axvline(-0.05, color='gray', linestyle='--', alpha=0.6)
ax.set_title("Compound Sentiment Score Distribution",
             fontsize=15, weight='bold')
ax.set_xlabel("VADER Compound Score (-1 = Negative, +1 = Positive)", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/03_compound_distribution.png", bbox_inches='tight')
plt.close()

# ---------- CHART 4 & 5 — Word Clouds (optional) ----------
if WORDCLOUD_AVAILABLE:
    def make_wc(text_list, color, filename, title):
        joined = " ".join(text_list)
        if not joined.strip():
            return
        wc = WordCloud(width=900, height=500, background_color='white',
                       colormap=color, stopwords=STOPWORDS,
                       max_words=100, collocations=False).generate(joined)
        plt.figure(figsize=(11, 6))
        plt.imshow(wc, interpolation='bilinear')
        plt.axis('off')
        plt.title(title, fontsize=15, weight='bold')
        plt.tight_layout()
        plt.savefig(f"{IMG_DIR}/{filename}", bbox_inches='tight')
        plt.close()

    make_wc(df[df['sentiment'] == 'Positive']['clean_text'].tolist(),
            'Greens', '04_wordcloud_positive.png', "Positive Word Cloud")
    make_wc(df[df['sentiment'] == 'Negative']['clean_text'].tolist(),
            'Reds', '05_wordcloud_negative.png', "Negative Word Cloud")
    print("✅ Word clouds generated.")
else:
    print("⏭️  Skipping word clouds — install 'wordcloud' to enable.")

# ---------- CHART 6 — Top N Words per Sentiment ----------
def top_words(texts, n=10):
    words = []
    for t in texts:
        words.extend([w for w in t.split() if w not in STOPWORDS and len(w) > 2])
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, sent, color in zip(axes, ['Positive', 'Negative'],
                           ['#2ecc71', '#e74c3c']):
    subset = df[df['sentiment'] == sent]['clean_text']
    top = top_words(subset, 10)
    if top:
        words, freqs = zip(*top)
        ax.barh(words[::-1], freqs[::-1], color=color)
        ax.set_title(f"Top Words — {sent}", fontsize=13, weight='bold')
        ax.set_xlabel("Frequency")
    else:
        ax.text(0.5, 0.5, "No data", ha='center', va='center')
        ax.set_title(f"Top Words — {sent}")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/06_top_words.png", bbox_inches='tight')
plt.close()

# ---------- CHART 7 — POS / NEU / NEG Averages ----------
sent_avg = df.groupby('sentiment')[['pos', 'neu', 'neg']].mean()
sent_avg = sent_avg.reindex(['Positive', 'Neutral', 'Negative']).fillna(0)
fig, ax = plt.subplots(figsize=(10, 5))
sent_avg.plot(kind='bar', ax=ax,
              color=['#2ecc71', '#95a5a6', '#e74c3c'], edgecolor='black')
ax.set_title("Average VADER Component Scores by Sentiment",
             fontsize=15, weight='bold')
ax.set_ylabel("Average Score", fontsize=12)
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.legend(title='Component')
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/07_component_scores.png", bbox_inches='tight')
plt.close()

# ---------- CHART 8 — Text Length vs Sentiment ----------
df['text_length'] = df['clean_text'].str.split().str.len()
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='sentiment', y='text_length',
            order=['Positive', 'Neutral', 'Negative'],
            palette=SENT_COLORS, ax=ax)
ax.set_title("Text Length by Sentiment", fontsize=15, weight='bold')
ax.set_xlabel("Sentiment")
ax.set_ylabel("Word Count")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/08_length_by_sentiment.png", bbox_inches='tight')
plt.close()

# ---------- CHART 9 — Dashboard ----------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Sentiment Analysis Dashboard", fontsize=18, weight='bold', y=1.02)

# A — Donut
axes[0, 0].pie(counts, labels=counts.index, autopct='%1.0f%%',
               colors=[SENT_COLORS[k] for k in counts.index],
               startangle=90,
               wedgeprops=dict(width=0.4, edgecolor='white'))
axes[0, 0].set_title("Sentiment Split", weight='bold')

# B — Compound histogram
axes[0, 1].hist(df['compound'], bins=30, color='#3498db', edgecolor='black')
axes[0, 1].axvline(0, color='red', linestyle='--', alpha=0.6)
axes[0, 1].set_title("Compound Score Distribution", weight='bold')
axes[0, 1].set_xlabel("Compound")

# C — Component averages
sent_avg.plot(kind='bar', ax=axes[1, 0],
              color=['#2ecc71', '#95a5a6', '#e74c3c'], edgecolor='black')
axes[1, 0].set_title("Average Component Scores", weight='bold')
axes[1, 0].set_xlabel("")
axes[1, 0].tick_params(axis='x', rotation=0)

# D — Length boxplot
sns.boxplot(data=df, x='sentiment', y='text_length',
            order=['Positive', 'Neutral', 'Negative'],
            palette=SENT_COLORS, ax=axes[1, 1])
axes[1, 1].set_title("Text Length by Sentiment", weight='bold')
axes[1, 1].set_xlabel("Sentiment")

plt.tight_layout()
plt.savefig(f"{IMG_DIR}/09_dashboard.png", bbox_inches='tight')
plt.close()

# ============================================================
# 8. SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("SENTIMENT ANALYSIS — COMPLETE")
print("=" * 60)

print("\n📊 Sentiment Distribution:")
for sent, cnt in counts.items():
    pct = cnt / len(df) * 100
    print(f"   {sent:10s}: {int(cnt):4d}  ({pct:5.1f}%)")

print(f"\n📈 Average Compound Score: {df['compound'].mean():.3f}")

print("\n🔝 Top 5 Positive Words:")
for w, f in top_words(df[df['sentiment'] == 'Positive']['clean_text'], 5):
    print(f"   • {w} ({f})")

print("\n🔻 Top 5 Negative Words:")
for w, f in top_words(df[df['sentiment'] == 'Negative']['clean_text'], 5):
    print(f"   • {w} ({f})")

print(f"\n✅ Charts saved to /{IMG_DIR}/:")
for f in sorted(os.listdir(IMG_DIR)):
    if f.endswith(".png"):
        print(f"   • {f}")

print("\n📌 Key Insights:")
print("   1. Sentiment distribution shows the overall tone of the dataset.")
print("   2. Positive texts cluster near compound = +0.6 to +1.0.")
print("   3. Negative texts cluster near compound = -0.6 to -1.0.")
print("   4. Neutral texts sit near 0 with low |compound| scores.")
print("   5. Frequent positive words highlight praise themes.")
print("   6. Frequent negative words highlight complaint themes.")
print("   7. Text length does NOT strongly correlate with sentiment polarity.")

print(f"\n✅ Full results saved to /{DATA_DIR}/sentiment_results.csv")

⚠ wordcloud NOT installed — word clouds will be skipped.
  To enable: pip install wordcloud

LOADING DATASET
⚠ URL load failed (HTTP Error 404: Not Found). Using built-in sample.
✅ Using built-in sample: 30 texts

TEXT CLEANING
Rows after cleaning: 30
                                                text  \
0  I absolutely love this product! Best purchase ...   
1  This is the worst experience I've ever had. Te...   
2  The movie was okay, nothing special but not ba...   
3  Amazing quality and fast delivery. Highly reco...   
4  Disappointed with the customer support. Very r...   

                                          clean_text  
0  i absolutely love this product best purchase ever  
1  this is the worst experience i ve ever had ter...  
2  the movie was okay nothing special but not bad...  
3  amazing quality and fast delivery highly recom...  
4   disappointed with the customer support very rude  

SENTIMENT SCORING
sentiment
Positive    18
Negative    10
Neutral      2
Name: c